# Session 1 — Analysis: Batch Size Scaling

## Background

The GPU executes eagerly on thousands of SIMT cores; each step dispatches work immediately. The TPU's systolic array (MXU) processes large matrix blocks in a fixed pipeline — it is most efficient when there is enough work to fill every cell. Batch size is the simplest lever that controls how much work arrives per step and therefore which execution model wins. Session 1 asks: where do these two devices diverge, and by how much?

## Goal

At the end of this session, participants will have run the same training loop on CPU, GPU, and TPU without changing model code. They will be able to read a throughput curve and explain why the GPU saturates early while the TPU scales near-linearly — grounding the interpretation in memory bandwidth and MXU utilization, not hardware marketing.

## Question

At what batch size does the TPU overtake the GPU, and how large does the gap become?

---

**Prerequisites:** Run `01-benchmark-cpu.ipynb`, `02-benchmark-gpu.ipynb`, and `03-benchmark-tpu.ipynb` first to populate `results/cpu.json`, `results/gpu.json`, and `results/tpu.json`.

In [1]:
import json, os

def load_results(results_dir="results"):
    out = {}
    if not os.path.isdir(results_dir):
        return out
    for fname in os.listdir(results_dir):
        if not fname.endswith(".json"):
            continue
        with open(os.path.join(results_dir, fname)) as f:
            data = json.load(f)
        hw = data.get("hardware", fname.replace(".json", "").upper())
        out[hw] = {
            "device_name": data.get("device_name", ""),
            "results":     {int(k): v for k, v in data["results"].items()},
        }
    return out

data = load_results("results")

CPU_RESULTS = data.get("CPU", {}).get("results", {})
GPU_RESULTS = data.get("GPU", {}).get("results", {})
TPU_RESULTS = data.get("TPU", {}).get("results", {})

ALL = {"CPU": CPU_RESULTS, "GPU": GPU_RESULTS, "TPU": TPU_RESULTS}
print("Loaded:", {hw: len(r) for hw, r in ALL.items() if r}, "batch sizes")

Loaded: {'CPU': 9, 'GPU': 9, 'TPU': 9} batch sizes


In [2]:
from itertools import chain

hw_order = [hw for hw in ["CPU", "GPU", "TPU"] if ALL.get(hw)]
all_bs   = sorted(set(chain.from_iterable(ALL[hw] for hw in hw_order)))
col      = 20

header = f"{'Batch':>8}" + "".join(f" | {hw + ' (samples/s)':>{col}}" for hw in hw_order)
print(header)
print("-" * len(header))
for bs in all_bs:
    row = f"{bs:>8}"
    for hw in hw_order:
        v = ALL[hw].get(bs)
        row += f" | {f'{v:,.0f}' if v else '—':>{col}}"
    print(row)

   Batch |      CPU (samples/s) |      GPU (samples/s) |      TPU (samples/s)
-----------------------------------------------------------------------------
       4 |                  148 |                1,388 |                  370
       8 |                  175 |                2,521 |                  745
      16 |                  164 |                2,866 |                1,496
      32 |                  192 |                2,895 |                2,984
      64 |                  204 |                2,760 |                5,968
     128 |                  174 |                2,688 |               11,900
     256 |                  143 |                2,653 |               23,862
     512 |                  147 |                2,618 |               47,580
    1024 |                  149 |                2,575 |               94,795


In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

COLORS  = {"CPU": "#888888", "GPU": "#76b900", "TPU": "#4285F4"}
MARKERS = {"CPU": "s",       "GPU": "o",       "TPU": "^"}

fig, ax = plt.subplots(figsize=(9, 5))

for hw in hw_order:
    xs = sorted(ALL[hw])
    ys = [ALL[hw][x] for x in xs]
    ax.plot(xs, ys, marker=MARKERS[hw], linewidth=2, markersize=7,
            color=COLORS[hw], label=hw)

ax.set_xscale("log", base=2)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.set_xlabel("Batch size", fontsize=12)
ax.set_ylabel("Throughput (samples / second)", fontsize=12)
ax.set_title("Throughput vs Batch Size — TransformerBlock (d=512)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which="both", linestyle="--", alpha=0.4)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()

os.makedirs("../docs/assets", exist_ok=True)
plt.savefig("../docs/assets/session_1_chart_throughput.png", dpi=150)
print("Saved session_1_chart_throughput.png")

Saved session_1_chart_throughput.png


In [4]:
if "GPU" not in hw_order or "TPU" not in hw_order:
    print("Need both GPU and TPU results for the crossover chart.")
else:
    common = sorted(set(GPU_RESULTS) & set(TPU_RESULTS))
    crossover = next((bs for bs in common if TPU_RESULTS[bs] > GPU_RESULTS[bs]), None)

    fig, ax = plt.subplots(figsize=(10, 5))
    for hw in ["GPU", "TPU"]:
        xs = sorted(ALL[hw])
        ys = [ALL[hw][x] for x in xs]
        ax.plot(xs, ys, marker=MARKERS[hw], linewidth=2, markersize=7,
                color=COLORS[hw], label=hw)

    if crossover:
        ax.axvline(x=crossover, color="red", linestyle=":", linewidth=1.5,
                   label=f"Crossover ≈ batch {crossover}")
        print(f"TPU overtakes GPU at batch_size ≈ {crossover}")
    else:
        print("No crossover in the measured range.")

    ax.set_xscale("log", base=2)
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.set_xlabel("Batch size", fontsize=12)
    ax.set_ylabel("Throughput (samples / second)", fontsize=12)
    ax.set_title("GPU vs TPU — Scaling and Crossover", fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, which="both", linestyle="--", alpha=0.4)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    plt.tight_layout()

    plt.savefig("../docs/assets/session_1_chart_crossover.png", dpi=150)
    print("Saved session_1_chart_crossover.png")

TPU overtakes GPU at batch_size ≈ 32


Saved session_1_chart_crossover.png


In [5]:
# Chart: TPU / GPU speedup multiplier across batch sizes
# Shows how the advantage grows from 0.27× at batch=4 to 38× at batch=1024
if "GPU" not in hw_order or "TPU" not in hw_order:
    print("Need both GPU and TPU results.")
else:
    common = sorted(set(GPU_RESULTS) & set(TPU_RESULTS))
    ratios = [TPU_RESULTS[bs] / GPU_RESULTS[bs] for bs in common]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(common, ratios, marker="^", linewidth=2.5, markersize=8, color="#4285F4")
    ax.axhline(1.0, color="red", linestyle=":", linewidth=1.5, label="parity  (1×  —  equal speed)")
    ax.fill_between(common, 1.0, ratios,
                    where=[r >= 1 for r in ratios], alpha=0.12, color="#4285F4",
                    label="TPU faster")
    ax.fill_between(common, ratios, 1.0,
                    where=[r < 1 for r in ratios], alpha=0.12, color="#76b900",
                    label="GPU faster")

    # Annotate peak multiplier
    max_r  = max(ratios)
    max_bs = common[ratios.index(max_r)]
    ax.annotate(f"{max_r:.0f}×", xy=(max_bs, max_r),
                xytext=(-40, 6), textcoords="offset points",
                fontsize=12, color="#4285F4", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#4285F4", lw=1.4))

    ax.set_xscale("log", base=2)
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.set_xlabel("Batch size", fontsize=12)
    ax.set_ylabel("TPU / GPU throughput ratio", fontsize=12)
    ax.set_title("TPU Speedup over GPU — from slower to 38× faster as batch grows", fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, which="both", linestyle="--", alpha=0.4)
    plt.tight_layout()

    plt.savefig("../docs/assets/session_1_chart_speedup.png", dpi=150)
    print("Saved session_1_chart_speedup.png")
    print()
    for bs, r in zip(common, ratios):
        tag = "  ← crossover" if abs(r - 1.0) < 0.15 else ""
        print(f"  batch={bs:<5}: {r:5.2f}×{tag}")

Saved session_1_chart_speedup.png

  batch=4    :  0.27×
  batch=8    :  0.30×
  batch=16   :  0.52×
  batch=32   :  1.03×  ← crossover
  batch=64   :  2.16×
  batch=128  :  4.43×
  batch=256  :  9.00×
  batch=512  : 18.17×
  batch=1024 : 36.82×


## Interpreting the results

| Observation | What it means |
|---|---|
| GPU leads at small batch sizes | Flexible dispatch, no XLA compilation overhead |
| TPU climbs steeply with batch size | Systolic array saturates; dense matrix throughput dominates |
| Crossover point | Batch size where TPU becomes the better choice for this workload |

## Hardware Decision Framework

```
Dense matrix multiplications dominate? (Transformer attention, large FFN)
  YES → TPUs often win at scale. Use PyTorch/XLA or JAX.
  NO  → Dynamic control flow / sparse ops / FP64 / small batches?
           YES → Stay on GPU. Safer and more flexible.
```